<a href="https://colab.research.google.com/github/mustapqilhasbi/Portfolio/blob/main/Stock_AI_Dashboard_Predictive_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 - Install dependencies
!pip install yfinance prophet plotly pandas numpy scikit-learn groq -q
print("Install selesai!")

Install selesai!


In [ ]:
# Cell 2 - Import & ambil data saham
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from groq import Groq
from google.colab import userdata

# Setup Groq (pakai secret yang sama)
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)

def get_stock_data(ticker, period="6mo"):
    stock = yf.Ticker(ticker)
    df = stock.history(period=period)
    df.reset_index(inplace=True)
    info = stock.info
    return df, info

# Test ambil data
print("Mengambil data BBCA.JK (Bank BCA)...")
df_bbca, info_bbca = get_stock_data("BBCA.JK")
print(f"Nama: {info_bbca.get('longName', 'N/A')}")
print(f"Harga terakhir: Rp {df_bbca['Close'].iloc[-1]:,.0f}")
print(f"Total data: {len(df_bbca)} hari")

print("\nMengambil data AAPL (Apple)...")
df_aapl, info_aapl = get_stock_data("AAPL")
print(f"Nama: {info_aapl.get('longName', 'N/A')}")
print(f"Harga terakhir: ${df_aapl['Close'].iloc[-1]:,.2f}")
print(f"Total data: {len(df_aapl)} hari")

print("\nData berhasil diambil!")

Mengambil data BBCA.JK (Bank BCA)...
Nama: PT Bank Central Asia Tbk
Harga terakhir: Rp 6,475
Total data: 118 hari

Mengambil data AAPL (Apple)...
Nama: Apple Inc.
Harga terakhir: $258.86
Total data: 124 hari

Data berhasil diambil!


In [ ]:
# Cell 3 - Candlestick Chart Interaktif
def plot_candlestick(df, ticker, currency="Rp"):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        vertical_spacing=0.03,
                        row_heights=[0.7, 0.3])

    fig.add_trace(go.Candlestick(
        x=df['Date'], open=df['Open'], high=df['High'],
        low=df['Low'], close=df['Close'], name=ticker,
        increasing_line_color='#26a69a',
        decreasing_line_color='#ef5350'
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=df['Date'], y=df['Volume'],
        name='Volume', marker_color='#7986cb'
    ), row=2, col=1)

    # Moving averages
    df['MA20'] = df['Close'].rolling(20).mean()
    df['MA50'] = df['Close'].rolling(50).mean()
    fig.add_trace(go.Scatter(x=df['Date'], y=df['MA20'],
        name='MA20', line=dict(color='orange', width=1.5)), row=1, col=1)
    fig.add_trace(go.Scatter(x=df['Date'], y=df['MA50'],
        name='MA50', line=dict(color='blue', width=1.5)), row=1, col=1)

    fig.update_layout(
        title=f'{ticker} - Stock Price Dashboard',
        yaxis_title=f'Harga ({currency})',
        xaxis_rangeslider_visible=False,
        template='plotly_dark', height=600
    )
    fig.show()

# Plot BBCA dan AAPL
plot_candlestick(df_bbca, "BBCA.JK", "Rp")
plot_candlestick(df_aapl, "AAPL", "$")
print("Chart berhasil dibuat!")

Chart berhasil dibuat!


In [ ]:
# Cell 4 - AI Stock Analyst
def analyze_stock(df, ticker, info):
    # Hitung indikator teknikal
    df['MA20'] = df['Close'].rolling(20).mean()
    df['MA50'] = df['Close'].rolling(50).mean()
    df['RSI'] = compute_rsi(df['Close'])

    current_price = df['Close'].iloc[-1]
    ma20 = df['MA20'].iloc[-1]
    ma50 = df['MA50'].iloc[-1]
    rsi = df['RSI'].iloc[-1]
    change_pct = ((df['Close'].iloc[-1] - df['Close'].iloc[-5]) / df['Close'].iloc[-5]) * 100

    prompt = f"""Kamu adalah analis saham profesional. Analisis saham berikut:

Ticker: {ticker}
Harga saat ini: {current_price:.2f}
MA20: {ma20:.2f}
MA50: {ma50:.2f}
RSI: {rsi:.2f}
Perubahan 5 hari: {change_pct:.2f}%

Berikan analisis singkat dan rekomendasi: BELI, JUAL, atau TAHAN.
Format:
- Kondisi teknikal
- Sinyal yang terlihat
- Rekomendasi + alasan singkat"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=400
    )
    return response.choices[0].message.content

def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0).rolling(period).mean()
    loss = -delta.where(delta < 0, 0).rolling(period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

# Analisis kedua saham
print("="*50)
print("ANALISIS AI - BBCA.JK")
print("="*50)
print(analyze_stock(df_bbca, "BBCA.JK", info_bbca))

print("\n" + "="*50)
print("ANALISIS AI - AAPL")
print("="*50)
print(analyze_stock(df_aapl, "AAPL", info_aapl))

ANALISIS AI - BBCA.JK
- Kondisi teknikal: Saat ini, harga saham BBCA.JK berada di bawah rata-rata pergerakan harga yang lebih tinggi (MA20 dan MA50), menunjukkan tekanan jual yang lebih kuat daripada tekanan beli. RSI pada 39,09 menunjukkan bahwa saham sedang dalam kondisi oversold, yang berpotensi memicu pembelian kembali.

- Sinyal yang terlihat: Sinyal oversold dari RSI dan perubahan harga yang positif dalam 5 hari terakhir menandakan potensi rebound. Namun, posisi harga di bawah MA20 dan MA50 masih menunjukkan sentimen bearish.

- Rekomendasi: TAHAN, karena meskipun ada sinyal oversold yang berpotensi memicu pembelian kembali, kondisi umum saham masih terlihat bearish sehingga tidak ada sinyal kuat untuk pembelian atau penjualan yang agresif saat ini. Rekomendasi ini juga didasarkan pada perubahan harga yang relatif stabil dalam 5 hari terakhir, menandakan bahwa saham sedang mencari titik equilibirum.

ANALISIS AI - AAPL
- Kondisi teknikal: Saat ini, harga saham AAPL berada di bawa

In [ ]:
# Cell 5 - Prediksi Harga dengan Prophet
from prophet import Prophet

def predict_stock(df, ticker, days=30):
    print(f"Memprediksi harga {ticker} untuk {days} hari ke depan...")

    # Siapkan data untuk Prophet
    df_prophet = df[['Date', 'Close']].copy()
    df_prophet.columns = ['ds', 'y']
    df_prophet['ds'] = pd.to_datetime(df_prophet['ds']).dt.tz_localize(None)

    # Train model
    model = Prophet(daily_seasonality=False,
                   weekly_seasonality=True,
                   yearly_seasonality=True,
                   changepoint_prior_scale=0.05)
    model.fit(df_prophet)

    # Prediksi
    future = model.make_future_dataframe(periods=days)
    forecast = model.predict(future)

    # Plot hasil prediksi
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_prophet['ds'], y=df_prophet['y'],
        name='Harga Aktual', line=dict(color='white', width=2)))
    fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat'],
        name='Prediksi', line=dict(color='orange', width=2, dash='dash')))
    fig.add_trace(go.Scatter(
        x=pd.concat([forecast['ds'], forecast['ds'][::-1]]),
        y=pd.concat([forecast['yhat_upper'], forecast['yhat_lower'][::-1]]),
        fill='toself', fillcolor='rgba(255,165,0,0.2)',
        line=dict(color='rgba(255,255,255,0)'), name='Range Prediksi'))

    last_actual = df_prophet['y'].iloc[-1]
    last_pred = forecast['yhat'].iloc[-1]
    change = ((last_pred - last_actual) / last_actual) * 100

    fig.update_layout(
        title=f'{ticker} - Prediksi {days} Hari ke Depan (Perubahan: {change:+.1f}%)',
        template='plotly_dark', height=500,
        yaxis_title='Harga', xaxis_title='Tanggal'
    )
    fig.show()
    print(f"Prediksi {days} hari: {change:+.1f}% dari harga sekarang")
    return forecast

# Prediksi BBCA dan AAPL
forecast_bbca = predict_stock(df_bbca, "BBCA.JK", days=30)
forecast_aapl = predict_stock(df_aapl, "AAPL", days=30)

Memprediksi harga BBCA.JK untuk 30 hari ke depan...


Prediksi 30 hari: +23.8% dari harga sekarang
Memprediksi harga AAPL untuk 30 hari ke depan...


Prediksi 30 hari: -39.0% dari harga sekarang


In [ ]:
# Cell 6 - Final Investment Summary
print("="*60)
print("   STOCK AI DASHBOARD - INVESTMENT SUMMARY REPORT")
print("="*60)

stocks = [
    (df_bbca, "BBCA.JK", info_bbca, forecast_bbca, "Rp"),
    (df_aapl, "AAPL", info_aapl, forecast_aapl, "$")
]

for df, ticker, info, forecast, currency in stocks:
    current = df['Close'].iloc[-1]
    pred_30 = forecast['yhat'].iloc[-1]
    change = ((pred_30 - current) / current) * 100
    high_52 = df['High'].max()
    low_52 = df['Low'].min()

    print(f"\n{'='*40}")
    print(f"  {ticker} — {info.get('longName','N/A')}")
    print(f"{'='*40}")
    print(f"  Harga sekarang : {currency}{current:,.2f}")
    print(f"  Prediksi 30hr  : {currency}{pred_30:,.2f} ({change:+.1f}%)")
    print(f"  52w High       : {currency}{high_52:,.2f}")
    print(f"  52w Low        : {currency}{low_52:,.2f}")
    print(f"  Sinyal AI      : {'BULLISH' if change > 0 else 'BEARISH'}")

print(f"\n{'='*60}")
print("  DISCLAIMER: Bukan saran investasi. Untuk tujuan edukasi.")
print(f"{'='*60}")

   STOCK AI DASHBOARD - INVESTMENT SUMMARY REPORT

  BBCA.JK — PT Bank Central Asia Tbk
  Harga sekarang : Rp6,475.00
  Prediksi 30hr  : Rp8,018.06 (+23.8%)
  52w High       : Rp8,327.97
  52w Low        : Rp6,107.63
  Sinyal AI      : BULLISH

  AAPL — Apple Inc.
  Harga sekarang : $258.86
  Prediksi 30hr  : $157.97 (-39.0%)
  52w High       : $288.35
  52w Low        : $243.19
  Sinyal AI      : BEARISH

  DISCLAIMER: Bukan saran investasi. Untuk tujuan edukasi.
